# Сверка ИНН: бизнес vs новая CRM (05.06.2026)

Тетрадка сверяет ИНН между:
- бизнес-выгрузкой `dbd_def_23_25.csv`;
- новой CRM-выгрузкой `дефолт_ИНН_05_06_2026.csv`.

Результат:
- сводные метрики пересечения;
- полный список ИНН из бизнеса, отсутствующих в новой CRM.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


def normalize_inn(value):
    """Нормализация ИНН без потери ведущих нулей.
    - оставляем только цифры;
    - 10/12 знаков оставляем;
    - 9 -> дополняем до 10;
    - 11 -> дополняем до 12;
    - прочие длины считаем невалидными (вернется None).
    """
    if pd.isna(value):
        return None

    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]

    digits = "".join(ch for ch in s if ch.isdigit())
    if not digits:
        return None

    if len(digits) in (10, 12):
        return digits
    if len(digits) == 9:
        return digits.zfill(10)
    if len(digits) == 11:
        return digits.zfill(12)

    return None


def pick_inn_col(df: pd.DataFrame, label: str) -> str:
    aliases = ["inn", "x_inn", "инн"]
    low_map = {c.lower().strip(): c for c in df.columns}
    for a in aliases:
        if a in low_map:
            return low_map[a]
    raise KeyError(f"В источнике {label} не найдена колонка ИНН (ожидались inn/x_inn/ИНН)")


def read_csv_robust(path: Path) -> pd.DataFrame:
    """Устойчивое чтение CSV: авторазделитель + fallback на ; и ,"""
    try:
        return pd.read_csv(path, dtype=str, sep=None, engine="python")
    except Exception:
        pass
    try:
        return pd.read_csv(path, dtype=str, sep=";")
    except Exception:
        return pd.read_csv(path, dtype=str, sep=",")


def find_business_file() -> Path:
    candidates = [
        Path("/home/jovyan/documents/DMUKP_FP_DEF/data"),
        Path("e:/DMUKP/data"),
        Path("e:/DMUKP/sources"),
        Path("e:/DMUKP"),
        Path.cwd(),
    ]
    filename = "dbd_def_23_25.csv"
    for base in candidates:
        p = base / filename
        if p.exists():
            return p
    for base in candidates:
        if base.exists():
            found = list(base.rglob(filename))
            if found:
                return found[0]
    raise FileNotFoundError("Не найден файл dbd_def_23_25.csv")


In [ ]:
business_path = find_business_file()
crm_path = business_path.parent / "дефолт_ИНН_05_06_2026.csv"

if not crm_path.exists():
    raise FileNotFoundError(
        f"Не найден файл новой CRM рядом с бизнес-выгрузкой: {crm_path}"
    )

print("Файлы сверки:")
print(f"- Бизнес: {business_path}")
print(f"- CRM:    {crm_path}")

df_business = read_csv_robust(business_path)
df_crm = read_csv_robust(crm_path)

df_business.columns = df_business.columns.str.strip()
df_crm.columns = df_crm.columns.str.strip()

business_inn_col = pick_inn_col(df_business, "dbd_def_23_25.csv")
crm_inn_col = pick_inn_col(df_crm, "дефолт_ИНН_05_06_2026.csv")

df_business["inn_raw"] = df_business[business_inn_col]
df_crm["inn_raw"] = df_crm[crm_inn_col]

df_business["inn"] = df_business["inn_raw"].apply(normalize_inn)
df_crm["inn"] = df_crm["inn_raw"].apply(normalize_inn)

invalid_business = df_business[df_business["inn"].isna()][["inn_raw"]].drop_duplicates().reset_index(drop=True)
invalid_crm = df_crm[df_crm["inn"].isna()][["inn_raw"]].drop_duplicates().reset_index(drop=True)

business_set = set(df_business["inn"].dropna().unique())
crm_set = set(df_crm["inn"].dropna().unique())

intersect = business_set & crm_set
business_only = business_set - crm_set
crm_only = crm_set - business_set


In [ ]:
base_business = len(business_set) if len(business_set) > 0 else 1

summary = pd.DataFrame([
    {"metric": "Уникальных ИНН в бизнесе", "value": len(business_set), "share_from_business": len(business_set) / base_business},
    {"metric": "Уникальных ИНН в новой CRM", "value": len(crm_set), "share_from_business": len(crm_set) / base_business},
    {"metric": "ИНН в пересечении", "value": len(intersect), "share_from_business": len(intersect) / base_business},
    {"metric": "ИНН только в бизнесе (нет в CRM)", "value": len(business_only), "share_from_business": len(business_only) / base_business},
    {"metric": "ИНН только в CRM", "value": len(crm_only), "share_from_business": len(crm_only) / base_business},
])

duplicates_control = pd.DataFrame([
    {"source": "business", "rows": len(df_business), "unique_inn": len(business_set), "duplicate_rows_by_inn": int(len(df_business) - df_business['inn'].nunique(dropna=True))},
    {"source": "crm", "rows": len(df_crm), "unique_inn": len(crm_set), "duplicate_rows_by_inn": int(len(df_crm) - df_crm['inn'].nunique(dropna=True))},
])

missing_in_crm_df = pd.DataFrame({"inn": sorted(business_only)})

print("=" * 90)
print("СВОДКА СВЕРКИ ИНН: БИЗНЕС vs НОВАЯ CRM")
print("=" * 90)
display(summary)

print("\nКонтроль дублей и размеров источников:")
display(duplicates_control)

print("\nКонтроль качества нормализации:")
print(f"- Невалидных ИНН в бизнесе: {len(invalid_business):,}")
print(f"- Невалидных ИНН в CRM:     {len(invalid_crm):,}")
if len(invalid_business) > 0:
    print("\nПримеры невалидных ИНН в бизнесе (до 20):")
    display(invalid_business.head(20))
if len(invalid_crm) > 0:
    print("\nПримеры невалидных ИНН в CRM (до 20):")
    display(invalid_crm.head(20))

print("\n" + "=" * 90)
print("ПОЛНЫЙ СПИСОК ИНН ИЗ БИЗНЕСА, КОТОРЫХ НЕТ В НОВОЙ CRM")
print("=" * 90)
display(missing_in_crm_df)
